# ContextAssembler Investigation: Per-CVE Failure Analysis

Compares per-CVE prediction outcomes between:
- **ContextAssembler** (`context_assembler_binary`) — 400 samples, 345 CVEs, function-level assembled context
- **CVEFixes** (`cvefixes_python_matched`) — 345 CVEs, file-level before/after-fix code

Both datasets cover **the same 345 CVE IDs** (100% overlap). For each model and CVE, this notebook
categorises the outcome as:
- `context_hurts` — CA wrong, CV right (most interesting for debugging)
- `context_helps` — CA right, CV wrong
- `both_wrong` — both approaches fail
- `both_correct` — both approaches succeed (omitted from output files)

Output: one Markdown file per model × CVE (for non-`both_correct` cases) under
`benchmarks/case_comparisons/{model}/{category}/CVE-YYYY-NNNNN.md`.

In [15]:
import ast
import glob
import json
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

# ─── Configuration ────────────────────────────────────────────────────────────
SELECTED_MODELS = None   # None = all models, e.g. ["qwen35-4b", "qwen3-8b-thinking"]
OUTPUT_DIR = Path("benchmarks/case_comparisons")

PROJECT_ROOT = Path("__file__").resolve().parent.parent
RESULTS_FILE = (
    PROJECT_ROOT / "results" / "context_assembler_experiments"
    / "vllm_comparison" / "experiment_plan_results.json"
)
DATASET_DIR = PROJECT_ROOT / "datasets_processed" / "context_assembler"
RESULTS_DIR = PROJECT_ROOT / "results" / "context_assembler_experiments" / "vllm_comparison"

DS_SUBDIRS = {
    "context_assembler": "context_assembler_binary",
    "cvefixes":          "cvefixes_python_matched",
}
DS_FILES = {
    "context_assembler": DATASET_DIR / "context_assembler_binary.json",
    "cvefixes":          DATASET_DIR / "cvefixes_python_matched.json",
}

MODEL_DISPLAY = {
    "qwen3-4b-thinking":               "Qwen3-4B (thinking)",
    "qwen3-8b-thinking":               "Qwen3-8B (thinking)",
    "qwen35-4b-thinking":              "Qwen3.5-4B (thinking)",
    "qwen35-4b":                       "Qwen3.5-4B",
    "llama3.2-3B":                     "Llama 3.2 (3B)",
    "deepseek-r1-distill-qwen2.5-7b":  "DeepSeek-R1 (7B)",
    "gemma3-4b":                       "Gemma3 (4B)",
    "code-llama-7b-python":            "CodeLlama (7B)",
}

CATEGORIES_TO_WRITE = ["context_hurts", "context_helps", "both_wrong"]
OUTCOME_LABELS = {
    "context_hurts": "Context Hurts (CA wrong, CV right)",
    "context_helps": "Context Helps (CA right, CV wrong)",
    "both_wrong":    "Both Wrong",
    "both_correct":  "Both Correct",
}

print(f"Project root : {PROJECT_ROOT}")
print(f"Output dir   : {OUTPUT_DIR.resolve()}")

Project root : /Users/somen/Zavodi/unik/llm4codesec-framework
Output dir   : /Users/somen/Zavodi/unik/llm4codesec-framework/benchmarks/benchmarks/case_comparisons


## Accuracy Summary

In [16]:
def _dataset_key(path: str) -> str:
    if "cvefixes" in path:
        return "cvefixes"
    if "context_assembler" in path:
        return "context_assembler"
    raise ValueError(f"Unknown dataset path: {path}")


with open(RESULTS_FILE) as f:
    results_raw = json.load(f)

summary_rows = []
for exp in results_raw["experiments"]:
    if not exp["is_success"]:
        continue
    bi = exp["benchmark_info"]
    ms = exp["metrics"]["summary"]
    summary_rows.append({
        "model":         bi["model_name"],
        "dataset":       _dataset_key(bi["dataset_path"]),
        "accuracy":      ms["accuracy"],
        "recall":        ms["recall"],
        "specificity":   ms["specificity"],
        "f1":            ms["f1_score"],
        "total_samples": bi["total_samples"],
    })

summary_df = pd.DataFrame(summary_rows)

pivot = summary_df.pivot_table(
    index="model", columns="dataset",
    values=["accuracy", "recall", "specificity", "f1"],
)
# Flatten column names
pivot.columns = [f"{m}_{ds}" for m, ds in pivot.columns]
pivot["delta_acc (cv-ca)"] = (
    pivot.get("accuracy_cvefixes", 0) - pivot.get("accuracy_context_assembler", 0)
)
pivot = pivot.sort_values("delta_acc (cv-ca)", ascending=False)
pivot.index = [MODEL_DISPLAY.get(m, m) for m in pivot.index]
pivot.index.name = "Model"

display(
    pivot[["accuracy_context_assembler", "accuracy_cvefixes", "delta_acc (cv-ca)"]]
    .rename(columns={
        "accuracy_context_assembler": "Accuracy (CA)",
        "accuracy_cvefixes":          "Accuracy (CV)",
    })
    .round(3)
    .style
    .background_gradient(
        subset=["Accuracy (CA)", "Accuracy (CV)"],
        cmap="RdYlGn", vmin=0.4, vmax=0.7,
    )
    .background_gradient(
        subset=["delta_acc (cv-ca)"],
        cmap="RdYlGn", vmin=-0.15, vmax=0.15,
    )
    .format("{:.3f}")
)

,Accuracy (CA),Accuracy (CV),delta_acc (cv-ca)
Model,,,
Qwen3-4B (thinking),0.535,0.562,0.027
Llama 3.2 (3B),0.420,0.438,0.018
Qwen3-8B (thinking),0.505,0.492,-0.013
Qwen3.5-4B,0.460,0.400,-0.060
Gemma3 (4B),0.454,0.390,-0.064
Qwen3.5-4B (thinking),0.455,0.376,-0.079


## Load Datasets

In [17]:
# Build sample_id → CVE and per-dataset CVE → [samples] maps
sample_to_cve: dict[str, str] = {}          # {sample_id: cve_id}
ca_cve_samples: dict[str, list] = defaultdict(list)   # {cve_id: [sample_dicts]}
cv_cve_samples: dict[str, list] = defaultdict(list)   # {cve_id: [sample_dicts]}

with open(DS_FILES["context_assembler"]) as f:
    ca_ds = json.load(f)
for s in ca_ds["samples"]:
    cve = s["metadata"]["cve_id"]
    sample_to_cve[s["id"]] = cve
    ca_cve_samples[cve].append(s)

with open(DS_FILES["cvefixes"]) as f:
    cv_ds = json.load(f)
for s in cv_ds["samples"]:
    cve = s["metadata"]["cve_id"]
    sample_to_cve[s["id"]] = cve
    cv_cve_samples[cve].append(s)

SHARED_CVES = sorted(set(ca_cve_samples) & set(cv_cve_samples))

ca_multi = sum(1 for v in ca_cve_samples.values() if len(v) > 1)
cv_both  = sum(1 for v in cv_cve_samples.values() if len(v) == 2)

print(f"CA  : {len(ca_ds['samples'])} samples, {len(ca_cve_samples)} CVEs ({ca_multi} multi-sample CVEs)")
print(f"CV  : {len(cv_ds['samples'])} samples, {len(cv_cve_samples)} CVEs ({cv_both} with both safe+vuln)")
print(f"Shared CVEs: {len(SHARED_CVES)}")

CA  : 200 samples, 179 CVEs (19 multi-sample CVEs)
CV  : 196 samples, 179 CVEs (17 with both safe+vuln)
Shared CVEs: 179


## Load Per-Sample Predictions

In [18]:
# Discover available models from result directories
_all_models: set[str] = set()
for ds_subdir in DS_SUBDIRS.values():
    ds_path = RESULTS_DIR / ds_subdir
    if ds_path.exists():
        for d in ds_path.iterdir():
            if d.is_dir():
                _all_models.add(d.name)

if SELECTED_MODELS is not None:
    models = [m for m in SELECTED_MODELS if m in _all_models]
else:
    models = sorted(_all_models)

print(f"Models ({len(models)}): {models}")

# all_predictions[model][dataset][sample_id] = {predicted, true, response_text}
all_predictions: dict[str, dict[str, dict]] = {}

for model in models:
    all_predictions[model] = {}
    for ds_key, ds_subdir in DS_SUBDIRS.items():
        pattern = str(RESULTS_DIR / ds_subdir / model / "step_by_step" / "predictions_*.json")
        files = sorted(glob.glob(pattern))
        if not files:
            print(f"  [warn] no predictions: {pattern}")
            continue
        with open(files[-1]) as f:
            preds = json.load(f)
        all_predictions[model][ds_key] = {
            p["sample_id"]: {
                "predicted":     p["predicted_label"],
                "true":          p["true_label"],
                "response_text": p.get("response_text", ""),
            }
            for p in preds
        }
        print(f"  {model}/{ds_key}: {len(preds)} predictions")

Models (6): ['gemma3-4b', 'llama3.2-3B', 'qwen3-4b-thinking', 'qwen3-8b-thinking', 'qwen35-4b', 'qwen35-4b-thinking']
  gemma3-4b/context_assembler: 196 predictions
  gemma3-4b/cvefixes: 118 predictions
  llama3.2-3B/context_assembler: 200 predictions
  llama3.2-3B/cvefixes: 130 predictions
  qwen3-4b-thinking/context_assembler: 200 predictions
  qwen3-4b-thinking/cvefixes: 128 predictions
  qwen3-8b-thinking/context_assembler: 200 predictions
  qwen3-8b-thinking/cvefixes: 128 predictions
  qwen35-4b/context_assembler: 200 predictions
  qwen35-4b/cvefixes: 125 predictions
  qwen35-4b-thinking/context_assembler: 200 predictions
  qwen35-4b-thinking/cvefixes: 125 predictions


## Per-CVE Outcome Categorization

A CVE is **correct** for a dataset/model pair when **all** samples for that CVE are correctly
classified. For ContextAssembler CVEs that have multiple function-level samples, every sample
must be correctly labeled. For CVEFixes CVEs that have only one sample, that one sample must
be correct.

In [19]:
def _cve_correct(
    model: str,
    ds_key: str,
    cve: str,
    ds_cve_samples: dict,
) -> bool | None:
    """Return True if all samples for this CVE are correctly predicted; None if data missing."""
    preds = all_predictions.get(model, {}).get(ds_key, {})
    samples = ds_cve_samples.get(cve, [])
    if not samples:
        return None
    results = []
    for s in samples:
        if s["id"] not in preds:
            return None  # incomplete predictions
        p = preds[s["id"]]
        results.append(p["predicted"] == p["true"])
    return all(results)


# outcomes[model][cve] = "context_hurts" | "context_helps" | "both_correct" | "both_wrong"
outcomes: dict[str, dict[str, str]] = {}

for model in models:
    outcomes[model] = {}
    for cve in SHARED_CVES:
        ca_ok = _cve_correct(model, "context_assembler", cve, ca_cve_samples)
        cv_ok = _cve_correct(model, "cvefixes",          cve, cv_cve_samples)
        if ca_ok is None or cv_ok is None:
            continue
        if   not ca_ok and     cv_ok: outcomes[model][cve] = "context_hurts"
        elif     ca_ok and not cv_ok: outcomes[model][cve] = "context_helps"
        elif     ca_ok and     cv_ok: outcomes[model][cve] = "both_correct"
        else:                         outcomes[model][cve] = "both_wrong"

total_categorized = sum(len(v) for v in outcomes.values())
print(f"Categorized {total_categorized} (model, CVE) pairs")

Categorized 711 (model, CVE) pairs


## Summary Statistics

In [20]:
stat_rows = []
for model in models:
    cats = outcomes.get(model, {})
    if not cats:
        continue
    counts = {
        c: sum(1 for v in cats.values() if v == c)
        for c in ["context_hurts", "context_helps", "both_correct", "both_wrong"]
    }
    total = sum(counts.values())
    stat_rows.append({
        "Model":           MODEL_DISPLAY.get(model, model),
        "context_hurts":   counts["context_hurts"],
        "context_helps":   counts["context_helps"],
        "both_correct":    counts["both_correct"],
        "both_wrong":      counts["both_wrong"],
        "total_CVEs":      total,
        "hurts_%":         f"{counts['context_hurts'] / total:.0%}" if total else "-",
        "helps_%":         f"{counts['context_helps'] / total:.0%}" if total else "-",
    })

stats_df = pd.DataFrame(stat_rows).set_index("Model")
print("Per-CVE outcome counts (strict: all samples must be correct):")
display(stats_df)

Per-CVE outcome counts (strict: all samples must be correct):


,context_hurts,context_helps,both_correct,both_wrong,total_CVEs,hurts_%,helps_%
Model,,,,,,,
Gemma3 (4B),2,11,37,60,110,2%,10%
Llama 3.2 (3B),15,12,35,61,123,12%,10%
Qwen3-4B (thinking),26,20,39,36,121,21%,17%
Qwen3-8B (thinking),19,17,38,47,121,16%,14%
Qwen3.5-4B,10,13,33,62,118,8%,11%
Qwen3.5-4B (thinking),6,13,34,65,118,5%,11%


## Generate Comparison Markdown Files

Writes `benchmarks/case_comparisons/{model}/{category}/CVE-YYYY-NNNNN.md`.
Skips `both_correct` (uninteresting for debugging).

In [21]:
def _parse_description(desc_str: str) -> str:
    """Extract English description from the list-of-dicts string stored in metadata."""
    if not desc_str:
        return "N/A"
    try:
        parsed = ast.literal_eval(str(desc_str))
        if isinstance(parsed, list):
            for item in parsed:
                if isinstance(item, dict) and item.get("lang") == "en":
                    return item.get("value", "N/A")
            if parsed and isinstance(parsed[0], dict):
                return parsed[0].get("value", "N/A")
    except Exception:
        pass
    return str(desc_str)[:500]


def _safe_str(v) -> str:
    """Return str(v), replacing NaN/None with 'N/A'."""
    if v is None:
        return "N/A"
    try:
        if isinstance(v, float) and v != v:  # NaN check
            return "N/A"
    except Exception:
        pass
    s = str(v)
    return "N/A" if s.lower() in ("nan", "none", "") else s


def _outcome_symbol(model: str, cve: str, ds_key: str, ds_cve_samples: dict) -> str:
    """Return ✓ or ✗ (or — if data missing) for a model/CVE/dataset."""
    preds = all_predictions.get(model, {}).get(ds_key, {})
    samples = ds_cve_samples.get(cve, [])
    results = []
    for s in samples:
        if s["id"] in preds:
            p = preds[s["id"]]
            results.append(p["predicted"] == p["true"])
    if not results:
        return "—"
    return "✓" if all(results) else "✗"


def _cv_response(model: str, cve: str) -> str:
    """Return CVEFixes response_text for this CVE (prefer vulnerable sample)."""
    preds = all_predictions.get(model, {}).get("cvefixes", {})
    samples = cv_cve_samples.get(cve, [])
    for s in sorted(samples, key=lambda x: x["label"], reverse=True):  # vuln first
        if s["id"] in preds:
            return preds[s["id"]]["response_text"] or "(empty)"
    return "(no prediction)"


def generate_markdown(model: str, cve: str, category: str) -> str:
    ca_samples = ca_cve_samples.get(cve, [])
    cv_samples = cv_cve_samples.get(cve, [])

    # Metadata — prefer CVEFixes fields (richer: severity, cwe_id, published_date)
    cv_meta = cv_samples[0]["metadata"] if cv_samples else {}
    ca_meta = ca_samples[0]["metadata"] if ca_samples else {}

    cwe_list = (cv_samples[0].get("cwe_types") or ca_samples[0].get("cwe_types") or []) if (cv_samples or ca_samples) else []
    cwe = _safe_str(cwe_list[0] if cwe_list else None)
    if cwe == "N/A":
        cwe_num = ca_meta.get("cwe_number") or cv_meta.get("cwe_number")
        cwe = f"CWE-{cwe_num}" if cwe_num else "N/A"

    severity    = _safe_str(cv_meta.get("severity") or ca_meta.get("severity"))
    description = _parse_description(cv_meta.get("description") or ca_meta.get("description", ""))

    lines = [
        f"# {cve} | {OUTCOME_LABELS.get(category, category)}",
        "",
        "## Metadata",
        f"- **CWE**: {cwe}",
        f"- **Severity**: {severity}",
        f"- **Description**: {description}",
        "",
        "## Prediction Outcomes",
        "",
        "| Model | CVEFixes | ContextAssembler |",
        "|-------|----------|------------------|",
    ]
    for m in models:
        cv_sym = _outcome_symbol(m, cve, "cvefixes",          cv_cve_samples)
        ca_sym = _outcome_symbol(m, cve, "context_assembler", ca_cve_samples)
        marker = " ← this file" if m == model else ""
        lines.append(f"| {MODEL_DISPLAY.get(m, m)}{marker} | {cv_sym} | {ca_sym} |")

    # ── CVEFixes section ──────────────────────────────────────────────────────
    lines += ["", "---", "", "## CVEFixes Code (file-level)", ""]

    cv_vuln = next((s for s in cv_samples if s["label"] == 1), None)
    cv_safe = next((s for s in cv_samples if s["label"] == 0), None)

    if cv_vuln:
        lines += [
            "### Before fix (vulnerable)",
            "```python",
            str(cv_vuln["code"]),
            "```",
            "",
        ]
    if cv_safe:
        lines += [
            "### After fix (safe)",
            "```python",
            str(cv_safe["code"]),
            "```",
            "",
        ]

    cv_resp = _cv_response(model, cve)
    lines += [
        f"### CVEFixes LLM Reasoning ({MODEL_DISPLAY.get(model, model)})",
        "```",
        cv_resp,
        "```",
        "",
    ]

    # ── ContextAssembler section ──────────────────────────────────────────────
    lines += ["---", "", "## ContextAssembler Code (function-level assembled context)", ""]

    ca_preds = all_predictions.get(model, {}).get("context_assembler", {})
    for i, s in enumerate(ca_samples, 1):
        label_str = "vulnerable" if s["label"] == 1 else "safe"
        p = ca_preds.get(s["id"], {})
        pred_label = p.get("predicted")
        true_label = p.get("true")
        correct    = "✓" if (pred_label is not None and pred_label == true_label) else "✗"
        resp       = p.get("response_text") or "(no prediction)"

        lines += [
            f"### Sample {i} — {s['id']} (label: {label_str}, prediction: {correct})",
            "```python",
            str(s["code"]),
            "```",
            "",
            f"### ContextAssembler LLM Reasoning — Sample {i} ({MODEL_DISPLAY.get(model, model)})",
            "```",
            resp,
            "```",
            "",
        ]

    return "\n".join(lines)


# ── Write all files ───────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
file_count = 0

for model in models:
    for cve, category in outcomes.get(model, {}).items():
        if category not in CATEGORIES_TO_WRITE:
            continue
        out_path = OUTPUT_DIR / model / category / f"{cve}.md"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text(generate_markdown(model, cve, category), encoding="utf-8")
        file_count += 1

print(f"\nGenerated {file_count} files under {OUTPUT_DIR}/")
print()
for model in models:
    model_dir = OUTPUT_DIR / model
    if not model_dir.exists():
        continue
    for cat in CATEGORIES_TO_WRITE:
        n = len(list((model_dir / cat).glob("*.md"))) if (model_dir / cat).exists() else 0
        if n:
            print(f"  {model}/{cat}/: {n} files")


Generated 495 files under benchmarks/case_comparisons/

  gemma3-4b/context_hurts/: 2 files
  gemma3-4b/context_helps/: 11 files
  gemma3-4b/both_wrong/: 60 files
  llama3.2-3B/context_hurts/: 15 files
  llama3.2-3B/context_helps/: 12 files
  llama3.2-3B/both_wrong/: 61 files
  qwen3-4b-thinking/context_hurts/: 26 files
  qwen3-4b-thinking/context_helps/: 20 files
  qwen3-4b-thinking/both_wrong/: 36 files
  qwen3-8b-thinking/context_hurts/: 19 files
  qwen3-8b-thinking/context_helps/: 17 files
  qwen3-8b-thinking/both_wrong/: 47 files
  qwen35-4b/context_hurts/: 10 files
  qwen35-4b/context_helps/: 13 files
  qwen35-4b/both_wrong/: 62 files
  qwen35-4b-thinking/context_hurts/: 6 files
  qwen35-4b-thinking/context_helps/: 13 files
  qwen35-4b-thinking/both_wrong/: 65 files
